# LiViFuser pilot-5 sweep — Kaggle T4 x2

Select the **GPU T4 x2** accelerator before running. This notebook verifies the uploaded bundle, checks both GPUs, runs a non-scientific smoke benchmark, and then launches the frozen 120-result sweep with one fold worker per T4.

In [ ]:
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

input_root = Path('/kaggle/input')
archives = sorted(input_root.rglob('livifuser_kaggle_t4x2_*.zip'))
manifests = sorted(input_root.rglob('cloud_bundle_manifest.json'))
repository = Path('/kaggle/working/LiViFuser')
assert not repository.exists(), f'Refusing to overwrite existing {repository}'
if len(archives) == 1 and not manifests:
    with zipfile.ZipFile(archives[0]) as archive:
        archive.extractall('/kaggle/working')
    source = archives[0]
elif not archives and len(manifests) == 1:
    shutil.copytree(manifests[0].parent, repository)
    source = manifests[0].parent
else:
    attached = sorted(str(path) for path in input_root.iterdir())
    raise AssertionError(
        'Expected one ZIP or one extracted cloud_bundle_manifest.json; '
        f'archives={archives}, manifests={manifests}, attached={attached}'
    )
os.chdir(repository)
print('Prepared', source, 'at', repository)

In [ ]:
import numpy as np
import torch

assert torch.cuda.is_available(), 'CUDA is unavailable; enable GPU T4 x2'
assert torch.cuda.device_count() == 2, f'Expected 2 GPUs, found {torch.cuda.device_count()}'
gpu_names = [torch.cuda.get_device_name(index) for index in range(2)]
assert all('T4' in name for name in gpu_names), f'Expected T4 x2, found {gpu_names}'
runtime = {
    'python': sys.version,
    'numpy': np.__version__,
    'torch': torch.__version__,
    'cuda': torch.version.cuda,
    'gpus': gpu_names,
}
print(runtime)

In [ ]:
subprocess.run([sys.executable, 'scripts/verify_cloud_bundle.py'], check=True)
test_command = [
    sys.executable,
    '-m',
    'unittest',
    'tests.test_baseline_sweep_resume',
    'tests.test_cloud_bundle',
    'tests.test_pilot5_cv',
    'tests.test_model_variants',
]
subprocess.run(test_command, check=True)

In [ ]:
import runpy

if True:
  from pathlib import Path

  repository = Path("/kaggle/working/LiViFuser")
  smoke_output = Path("/kaggle/working/t4_smoke.json")

  assert not smoke_output.exists(), f"Refusing to overwrite {smoke_output}"

  torch.cuda.set_device(0)

  original_reset = torch.cuda.reset_peak_memory_stats
  original_max = torch.cuda.max_memory_allocated

  torch.cuda.reset_peak_memory_stats = lambda device=None: original_reset()
  torch.cuda.max_memory_allocated = lambda device=None: original_max()

  original_argv = sys.argv[:]

  sys.argv = [
      "benchmark_training_device.py",
      "--device",
      "cuda:0",
      "--steps",
      "100",
      "--output",
      str(smoke_output),
  ]

  try:
      runpy.run_path(
          str(repository / "scripts/benchmark_training_device.py"),
          run_name="__main__",
      )
  finally:
      sys.argv = original_argv
      torch.cuda.reset_peak_memory_stats = original_reset
      torch.cuda.max_memory_allocated = original_max

The next cell is the scientific run. It prints live fold/model progress and writes a checkpoint plus result after every completed model/seed. Do not edit the extracted repository while it runs.

In [ ]:
sweep_command = [
    sys.executable,
    'scripts/run_pilot5_cv.py',
    '--max-workers',
    '2',
    '--cuda-device',
    '0',
    '--cuda-device',
    '1',
]
subprocess.run(sweep_command, check=True)

In [ ]:
import hashlib
import shutil

result_root = (
    repository
    / 'artifacts/experiments/pilot5_leave_one_episode_out_kaggle_t4x2_v1'
)
archive_base = Path('/kaggle/working/pilot5_leave_one_episode_out_kaggle_t4x2_v1')
archive_path = Path(shutil.make_archive(str(archive_base), 'zip', result_root))
hasher = hashlib.sha256()
with archive_path.open('rb') as stream:
    for chunk in iter(lambda: stream.read(1024 * 1024), b''):
        hasher.update(chunk)
digest = hasher.hexdigest()
print({'result_archive': str(archive_path), 'bytes': archive_path.stat().st_size, 'sha256': digest})